In [311]:
import pandas as pd

In [312]:
data = pd.read_csv("C://Users/User/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [313]:
from IPython.display import display

# В Jupyter можно использовать display с настройками
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(data[:5])

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [314]:
data['gender'] = data['gender'].apply(lambda x: 1 if x == 'Male' else 0)

In [315]:
arr = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
       'StreamingTV', 'StreamingMovies', 'PaperlessBilling','Churn']
for col in arr:
    data[col] = data[col].apply(lambda x: 1 if x == 'Yes' else 0)

In [316]:
data = data.drop('customerID', axis=1)

In [317]:
arr = ['MonthlyCharges', 'TotalCharges', 'tenure']

for col in arr:
    data[col] = pd.to_numeric(data[col], errors='coerce')
    data[col] = data[col].apply(lambda x: data[col].mean() if pd.isna(x) else x)
    
for col in arr:
    data[col] = (data[col] - data[col].min())/(data[col].max() - data[col].min())

In [318]:
d = {
    'DSL': 1,
    'Fiber optic': 0.5,
    'No': 0
}

data['InternetService'] = data['InternetService'].apply(lambda x: d[x])

In [319]:
arr = ['Contract', 'PaymentMethod']

for col in arr:
    data[col] = data[col].astype('category').cat.codes

In [320]:
X = data.loc[:, :'TotalCharges']
Y = data['Churn']


In [321]:
cor_matrix = X.corr()


In [322]:
import numpy as np

In [323]:
ones = np.ones(cor_matrix.shape)
triu = np.triu(ones, k=1).astype(bool)
cor_matrix = cor_matrix.where(triu)

In [324]:
cor_matrix

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
gender,NaN,-0.001874,-0.001808,0.010517,0.005106,-0.006488,-0.008414,0.000863,-0.017021,-0.013773,-0.002105,-0.009212,-0.008393,-0.010487,0.000126,-0.011754,0.017352,-0.014569,0.000048
SeniorCitizen,NaN,NaN,0.016479,-0.211185,0.016567,0.008576,0.142948,0.032310,-0.038653,0.066572,0.059428,-0.060625,0.105378,0.120176,-0.142554,0.156530,-0.038551,0.220173,0.102395
Partner,NaN,NaN,NaN,0.452676,0.379697,0.017706,0.142057,-0.000891,0.143106,0.141498,0.153786,0.119999,0.124666,0.117412,0.294806,-0.014877,-0.154798,0.096848,0.318812
Dependents,NaN,NaN,NaN,NaN,0.159712,-0.001762,-0.024526,-0.044590,0.080972,0.023671,0.013963,0.063268,-0.016558,-0.039741,0.243187,-0.111377,-0.040292,-0.113890,0.064535
tenure,NaN,NaN,NaN,NaN,NaN,0.008448,0.331941,0.030359,0.327203,0.360277,0.360653,0.324221,0.279756,0.286111,0.671607,0.006152,-0.370436,0.247900,0.824757
PhoneService,NaN,NaN,NaN,NaN,NaN,NaN,0.279690,-0.387436,-0.092893,-0.052312,-0.071227,-0.096340,-0.022574,-0.032959,0.002247,0.016505,-0.004184,0.247398,0.112851
MultipleLines,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011124,0.098108,0.202237,0.201137,0.100571,0.257152,0.258751,0.107114,0.163530,-0.171026,0.490434,0.468689
InternetService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.393013,0.314737,0.306805,0.389382,0.242532,0.250343,-0.099721,0.138625,-0.086140,0.323260,0.175429
OnlineSecurity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.283832,0.275438,0.354931,0.176207,0.187398,0.245530,-0.003636,-0.150100,0.296594,0.412245
OnlineBackup,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.303546,0.294233,0.282106,0.274501,0.155085,0.126735,-0.170898,0.441780,0.509692


In [325]:
def cv(x):
    return x.std() / x.mean() * 100

for col in X.columns:
    print(f'{col}: {cv(X[col])}')
    

gender: 99.06021572705731
SeniorCitizen: 227.3320086086972
Partner: 103.46036796004911
Dependents: 152.91326586602074
tenure: 75.86842617906674
PhoneService: 32.74615576878069
MultipleLines: 117.0801830036162
InternetService: 65.4610686160132
OnlineSecurity: 157.75658257151727
OnlineBackup: 137.83384505065357
DeviceProtection: 138.13754986308956
TechSupport: 156.39827420388016
StreamingTV: 126.57016396540648
StreamingMovies: 125.62599014474895
Contract: 120.75134906730574
PaperlessBilling: 82.98564171853205
PaymentMethod: 67.8450262518455
MonthlyCharges: 64.69351147169772
TotalCharges: 100.02207184209139


In [326]:
X.describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.504756,0.162147,0.483033,0.299588,0.449599,0.903166,0.421837,0.563538,0.286668,0.344881,0.343888,0.290217,0.384353,0.387903,0.690473,0.592219,1.574329,0.462803,0.261309
std,0.500013,0.368612,0.499748,0.458110,0.341104,0.295752,0.493888,0.368898,0.452237,0.475363,0.475038,0.453895,0.486477,0.487307,0.833755,0.491457,1.068104,0.299403,0.261366
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.125000,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.171642,0.044245
50%,1.000000,0.000000,0.000000,0.000000,0.402778,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,0.518408,0.159445
75%,1.000000,0.000000,1.000000,1.000000,0.763889,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,0.712438,0.434780
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000


In [327]:
X.var()

gender              0.250013
SeniorCitizen       0.135875
Partner             0.249748
Dependents          0.209865
tenure              0.116352
PhoneService        0.087469
MultipleLines       0.243925
InternetService     0.136086
OnlineSecurity      0.204518
OnlineBackup        0.225970
DeviceProtection    0.225661
TechSupport         0.206020
StreamingTV         0.236659
StreamingMovies     0.237468
Contract            0.695148
PaperlessBilling    0.241530
PaymentMethod       1.140846
MonthlyCharges      0.089642
TotalCharges        0.068312
dtype: float64

In [328]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,0.013889,0,0,1.0,0,1,0,0,0,0,0,1,2,0.115423,0.001275
1,1,0,0,0,0.472222,1,0,1.0,1,0,1,0,0,0,1,0,3,0.385075,0.215867
2,1,0,0,0,0.027778,1,0,1.0,1,1,0,0,0,0,0,1,3,0.354229,0.010310
3,1,0,0,0,0.625000,0,0,1.0,1,0,1,1,0,0,1,0,0,0.239303,0.210241
4,0,0,0,0,0.027778,1,0,0.5,0,0,0,0,0,0,0,1,2,0.521891,0.015330


In [329]:
X.shape

(7043, 19)

In [330]:
from collections import defaultdict

count_cor = defaultdict(dict)

for idx in cor_matrix.index:
    for col in cor_matrix.columns:
        if pd.notna(cor_matrix.loc[idx, col]):
            if abs(cor_matrix.loc[idx, col]) >= 0.4:
                count_cor[idx][col] = cor_matrix.loc[idx, col]

count_cor

defaultdict(dict,
            {'Partner': {'Dependents': np.float64(0.4526762829294659)},
             'tenure': {'Contract': np.float64(0.6716065492280602),
              'TotalCharges': np.float64(0.8247573156351051)},
             'MultipleLines': {'MonthlyCharges': np.float64(0.4904339694400286),
              'TotalCharges': np.float64(0.4686893761134366)},
             'OnlineSecurity': {'TotalCharges': np.float64(0.41224461723017536)},
             'OnlineBackup': {'MonthlyCharges': np.float64(0.4417800943594724),
              'TotalCharges': np.float64(0.509691520884732)},
             'DeviceProtection': {'StreamingMovies': np.float64(0.4021105377661728),
              'MonthlyCharges': np.float64(0.4826920482732959),
              'TotalCharges': np.float64(0.5224618309296661)},
             'TechSupport': {'TotalCharges': np.float64(0.4324795954494656)},
             'StreamingTV': {'StreamingMovies': np.float64(0.5330938326943199),
              'MonthlyCharges': np.float6

In [331]:


for col in X.columns:
    null = 0
    one = 0
    for el in X[col]:
        if el == 0:
            null += 1
        if el == 1:
            one += 1
    print(f'{col} (null: {(null / len(X[col])) * 100}, one: {(one / len(X[col])) * 100})')


gender (null: 49.5243504188556, one: 50.4756495811444)
SeniorCitizen (null: 83.78531875621185, one: 16.21468124378816)
Partner (null: 51.696720147664344, one: 48.30327985233565)
Dependents (null: 70.04117563538264, one: 29.95882436461735)
tenure (null: 0.1561834445548772, one: 5.139855175351412)
PhoneService (null: 9.683373562402386, one: 90.31662643759762)
MultipleLines (null: 57.81627147522362, one: 42.18372852477638)
InternetService (null: 21.666903308249324, one: 34.37455629703251)
OnlineSecurity (null: 71.33323867670028, one: 28.666761323299728)
OnlineBackup (null: 65.51185574329122, one: 34.48814425670879)
DeviceProtection (null: 65.61124520800796, one: 34.38875479199205)
TechSupport (null: 70.97827630271192, one: 29.02172369728809)
StreamingTV (null: 61.564674144540675, one: 38.435325855459325)
StreamingMovies (null: 61.209711770552325, one: 38.790288229447675)
Contract (null: 55.01916796819537, one: 20.914383075394007)
PaperlessBilling (null: 40.77807752378248, one: 59.22192247

In [332]:
X = X.drop('Contract', axis=1)
cor_matrix = cor_matrix.drop('Contract', axis=1)
cor_matrix = cor_matrix.drop('Contract', axis=0)

In [333]:
X = X.drop('TotalCharges', axis=1)
cor_matrix = cor_matrix.drop('TotalCharges', axis=1)
cor_matrix = cor_matrix.drop('TotalCharges', axis=0)

In [334]:
X = X.drop('Dependents', axis=1)
cor_matrix = cor_matrix.drop('Dependents', axis=1)
cor_matrix = cor_matrix.drop('Dependents', axis=0)

In [335]:
X = X.drop('StreamingMovies', axis=1)
cor_matrix = cor_matrix.drop('StreamingMovies', axis=1)
cor_matrix = cor_matrix.drop('StreamingMovies', axis=0)

In [336]:
X = X.drop('MonthlyCharges', axis=1)
cor_matrix = cor_matrix.drop('MonthlyCharges', axis=1)
cor_matrix = cor_matrix.drop('MonthlyCharges', axis=0)

In [337]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size = 0.2,
    random_state = 42,
    shuffle = True
)




In [338]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [339]:
x0 = len([i for i in y_train if i == 0])
x1 = len([i for i in y_train if i == 1])
n = len(y_train)
print(f'0 : {(x0/n)*100}')
print(f'1 : {(x1/n)*100}')
print(n)

0 : 73.44692935747248
1 : 26.55307064252751
5634


In [340]:
binary_col = []

for col in X.columns:
    null = 0
    one = 0
    for el in X[col]:
        if el == 0:
            null += 1
        if el == 1:
            one += 1
            
    if (null / len(X[col])) + (one / len(X[col])) == 1 :
        binary_col.append(col)

In [356]:
from imblearn.over_sampling import SMOTENC


cat_indices = [X.columns.get_loc(c) for c in binary_col]

smote = SMOTENC(
    categorical_features=cat_indices,
    k_neighbors=5,
    sampling_strategy=1,
    random_state=42
    )
X_smote, y_smote = smote.fit_resample(X_train, y_train)


In [357]:
x0 = len([i for i in y_smote if i == 0])
x1 = len([i for i in y_smote if i == 1])
n = len(y_smote)
print(f'0 : {(x0/n)*100}')
print(f'1 : {(x1/n)*100}')
print(n)

0 : 50.0
1 : 50.0
8276


In [343]:
def TP(predict, real):
    c = 0
    for i in range(len(predict)):
        if predict[i] == real.iloc[i] == 1:
            c += 1
    return c

def TN(predict, real):
    c = 0
    for i in range(len(predict)):
        if predict[i] == real.iloc[i]  == 0:
            c += 1
    return c

def FP(predict, real):
    c = 0
    for i in range(len(predict)):
        if predict[i] == 1 and real.iloc[i]  == 0:
            c += 1
    return c

def FN(predict, real):
    c = 0
    for i in range(len(predict)):
        if predict[i] == 0 and real.iloc[i]  == 1:
            c += 1
    return c

def accurasy(predict, real):
    if (TN(predict, real) + TP(predict, real) + FN(predict, real) + FP(predict, real)) == 0:
        return 0
    return (TN(predict, real) + TP(predict, real)) / (TN(predict, real) + TP(predict, real) + FN(predict, real) + FP(predict, real))

def recall(predict, real):
    if (TP(predict, real) + FN(predict, real)) == 0:
        return 0
    return TP(predict, real) / (TP(predict, real) + FN(predict, real))

def precision(predict, real):
    if (TP(predict, real) + FP(predict, real)) == 0:
        return 0
    return TP(predict, real) / (TP(predict, real) + FP(predict, real))

def F1(predict, real):
    if recall(predict, real) == 0 or precision(predict, real) == 0:
        return 0
    return 2 / (1/recall(predict, real) + 1/precision(predict, real))

def specifity(predict, real):
    if (TN(predict, real) + FP(predict, real)) == 0:
        return 0
    return TN(predict, real) / (TN(predict, real) + FP(predict, real))

In [358]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier()

dummy_model.fit(X_train, y_train)
predict = dummy_model.predict(X_test)


In [359]:
print(f'accurasy: {accurasy(predict, y_test)}')
print(f'recall: {recall(predict, y_test)}')
print(f'precision: {precision(predict, y_test)}')
print(f'F1: {F1(predict, y_test)}')
print(f'specifity: {specifity(predict, y_test)}')

accurasy: 0.7352732434350603
recall: 0.0
precision: 0
F1: 0
specifity: 1.0


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(
    penalty='l1',
    C=0.1,
    solver='saga',
    class_weight='balanced',
    
    max_iter=1000,
    )

logreg_model.fit(X_smote, y_smote)
predict = logreg_model.predict(X_test)


c:\Users\User\Desktop\petproject\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\User\Desktop\petproject\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [373]:
y_proba = logreg_model.predict_proba(X_test)[: , 1]
tresh_hold = 0.5
predict = (y_proba >= tresh_hold).astype(int)

In [374]:
print('====SMOTE===')
print(f'accurasy: {accurasy(predict, y_test)}')
print(f'recall: {recall(predict, y_test)}')
print(f'precision: {precision(predict, y_test)}')
print(f'F1: {F1(predict, y_test)}')
print(f'specifity: {specifity(predict, y_test)}')

====SMOTE===
accurasy: 0.7437899219304471
recall: 0.7587131367292225
precision: 0.5108303249097473
F1: 0.610571736785329
specifity: 0.7384169884169884


In [349]:
logreg_model2 = LogisticRegression(

)

logreg_model2.fit(X_train, y_train)
predict = logreg_model2.predict(X_test)

In [350]:
print('====CLEAN===')
print(f'accurasy: {accurasy(predict, y_test)}')
print(f'recall: {recall(predict, y_test)}')
print(f'precision: {precision(predict, y_test)}')
print(f'F1: {F1(predict, y_test)}')
print(f'specifity: {specifity(predict, y_test)}')

====CLEAN===
accurasy: 0.7885024840312278
recall: 0.41823056300268097
precision: 0.6582278481012658
F1: 0.5114754098360655
specifity: 0.9218146718146718
